# 06 · Anomaly detection
Flag abnormal samples via reconstruction error and classical detectors; validate against recorded field events (`has_note`).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from dga.config import load_config, set_seed
cfg = load_config(); set_seed(cfg.seed)

In [ ]:
import numpy as np, torch
from dga.config import PROJECT_ROOT
from dga import data as dga_data
from dga.preprocessing import build_feature_matrix
from dga.models.autoencoder import Autoencoder
df = dga_data.load_clean(); fm = build_feature_matrix(df, cfg)
model = Autoencoder(fm.X.shape[1], tuple(cfg.autoencoder.hidden_dims), cfg.autoencoder.latent_dim)
model.load_state_dict(torch.load(PROJECT_ROOT/'results'/'models'/'autoencoder.pt'))
from dga import anomaly
scores = anomaly.reconstruction_scores(model, fm.X)
flags = anomaly.threshold_flags(scores, cfg.anomaly.threshold_quantile)
int(flags.sum())

## Score distribution and threshold

In [ ]:
from dga import viz
thr = np.quantile(scores, cfg.anomaly.threshold_quantile)
fig = viz.plot_reconstruction_error(scores, thr); fig

## Do flagged anomalies match real field events?

In [ ]:
has_note = df['has_note'].loc[fm.index].to_numpy()
anomaly.evaluate_against_notes(flags, has_note)

## Compare detectors
Run `anomaly.isolation_forest_scores(fm.X)` and `ocsvm_scores(fm.X)` and compare precision/recall vs notes.